# Deploy Graph Orchestrator to Amazon Bedrock AgentCore Runtime

This tutorial series builds a **5-agent e-commerce assistant** using **Strands Agents GraphBuilder** for deterministic routing across Amazon Bedrock AgentCore runtimes. In this notebook, you deploy the Graph Orchestrator -- the central control plane that coordinates all 4 specialist agents via a DAG (Directed Acyclic Graph).

**Notebook 5 of 5** -- Requires all 4 specialist agents (Classifier, Product, Order, Recommendation) deployed in Notebooks 1-4.

## Architecture Overview

The Graph Orchestrator runs on Runtime 5 and coordinates all 4 specialist agents via GraphBuilder:

| Runtime | Agent | Protocol | Tools |
|---------|-------|----------|-------|
| 1 | Classifier | A2A (port 9000) | None (pure LLM) |
| 2 | Product | A2A (port 9000) | HTTP API tools |
| 3 | Order | A2A (port 9000) | DynamoDB via MCP |
| 4 | Recommendation | A2A (port 9000) | None (LLM synthesis) |
| **5** | **Graph Orchestrator** | **HTTP (port 8080)** | **GraphBuilder + 4x A2A clients** |

### Graph Topology

```
classifier_node --> product_node         (BROWSE or RECOMMEND)
classifier_node --> order_node           (ORDER or RECOMMEND)
product_node    --> recommendation_node  (RECOMMEND only)
order_node      --> recommendation_node  (RECOMMEND only)
```

### 3 Execution Paths

| Intent | Path | Agents |
|--------|------|--------|
| BROWSE | Classifier -> Product | 2 |
| ORDER | Classifier -> Order | 2 |
| RECOMMEND | Classifier -> Product + Order (parallel) -> Recommendation | 4 |

### GraphBuilder Features Demonstrated

| Feature | How It's Used |
|---------|---------------|
| **Conditional routing** | Classifier's JSON output determines which downstream agents execute |
| **Parallel execution** | Product and Order agents run simultaneously for RECOMMEND intent |
| **Join/Fan-in** | Recommendation agent waits for both Product and Order to complete |
| **Chaining** | Recommendation receives combined context from predecessor nodes |

## Prerequisites

- [AWS CLI](https://aws.amazon.com/cli/) installed and configured
- Python 3.10 or higher
- Docker or Podman installed
- Claude Sonnet 4 model access in Amazon Bedrock
- **All 4 specialist agents deployed:**
  - Notebook 01: Classifier Agent URL stored in SSM
  - Notebook 02: Product Agent URL stored in SSM
  - Notebook 03: Order Agent URL stored in SSM (+ DynamoDB table created)
  - Notebook 04: Recommendation Agent URL stored in SSM

In [ ]:
import json
import os
from pathlib import Path
from urllib.parse import quote
from uuid import uuid4

import boto3

NOTEBOOK_DIR = Path.cwd()

from utils import (
    ORCHESTRATOR_AGENT_NAME,
    ORCHESTRATOR_ROLE_NAME,
    SSM_CLASSIFIER_AGENT_URL,
    SSM_PRODUCT_AGENT_URL,
    SSM_ORDER_AGENT_URL,
    SSM_RECOMMENDATION_AGENT_URL,
    create_agentcore_role,
    get_agent_url,
    store_agent_url,
)
from utils.streaming import GraphStreamingDisplay

session = boto3.Session()
region = session.region_name or "us-west-2"
account_id = boto3.client("sts").get_caller_identity()["Account"]

print(f"Region: {region}")
print(f"Account: {account_id}")
print(f"Agent Name: {ORCHESTRATOR_AGENT_NAME}")

### Verify Agent URLs

Confirm all 4 specialist agents are deployed and their URLs are stored in SSM Parameter Store.

In [ ]:
print("Verifying agent URLs from SSM Parameter Store...")
print("-" * 50)

ssm_params = {
    "Classifier": SSM_CLASSIFIER_AGENT_URL,
    "Product": SSM_PRODUCT_AGENT_URL,
    "Order": SSM_ORDER_AGENT_URL,
    "Recommendation": SSM_RECOMMENDATION_AGENT_URL,
}

all_ready = True
for name, param in ssm_params.items():
    try:
        url = get_agent_url(param_name=param, region=region)
        print(f"  {name}: {url[:80]}...")
    except Exception as e:
        print(f"  {name}: NOT FOUND - deploy this agent first")
        all_ready = False

if all_ready:
    print("\nAll agent URLs found. Ready to deploy Graph Orchestrator.")
else:
    print("\nSome agents are missing. Deploy them before continuing.")

---
## Step 1: Create Graph Orchestrator

The Graph Orchestrator consists of three files:
1. **`graph_agent.py`** -- GraphBuilder DAG definition with conditional edges and node agent factory
2. **`app.py`** -- BedrockAgentCoreApp HTTP entrypoint that streams graph execution events
3. **`sigv4_auth.py`** -- AWS SigV4 authentication for A2A calls to AgentCore runtimes

### How GraphBuilder Works

Each graph node is a local Strands Agent that uses `A2AClientToolProvider` to forward requests to a remote A2A agent on its own AgentCore runtime:

```
Graph Node (local Agent)
    |
    v
A2AClientToolProvider
    |
    v  (SigV4-signed HTTP)
Remote AgentCore Runtime
    |
    v
Remote A2A Agent (Classifier/Product/Order/Recommendation)
```

The GraphBuilder handles:
- **Execution order**: Topological sort determines which nodes run when
- **Conditional edges**: Condition functions evaluate `GraphState` to determine routing
- **Parallel execution**: Independent nodes in the same batch run concurrently
- **State passing**: `_build_node_input()` passes predecessor outputs to downstream nodes

### Graph Agent (DAG Definition)

The following cell creates `graph_orchestrator/graph_agent.py`:

| Section | What It Does |
|---------|---------------|
| Intent Parsing | Extracts BROWSE/ORDER/RECOMMEND from Classifier's JSON output |
| Condition Functions | Three boolean functions that determine edge traversal based on intent |
| Node Agent Factory | Creates local agents with `A2AClientToolProvider` for remote A2A calls |
| Graph Construction | Builds GraphBuilder DAG with 4 nodes, 4 conditional edges |

In [ ]:
%%writefile graph_orchestrator/graph_agent.py
"""Graph orchestrator for e-commerce multi-agent system.

Defines a GraphBuilder DAG that routes customer queries through 4 remote
agents deployed to Amazon Bedrock AgentCore:
1. Classifier: Determines intent (BROWSE, ORDER, RECOMMEND)
2. Product Agent: Searches product catalog via DummyJSON API
3. Order Agent: Looks up customer orders via DynamoDB MCP
4. Recommendation Agent: Generates personalized product recommendations

Graph topology:
  classifier_node -> product_node         (BROWSE or RECOMMEND)
  classifier_node -> order_node           (ORDER or RECOMMEND)
  product_node    -> recommendation_node  (RECOMMEND)
  order_node      -> recommendation_node  (RECOMMEND)

Execution paths:
  BROWSE:    Classifier -> Product (2 agents)
  ORDER:     Classifier -> Order (2 agents)
  RECOMMEND: Classifier -> Product + Order (parallel) -> Recommendation (4 agents)
"""

import logging
import re

import boto3
from strands import Agent
from strands.models import BedrockModel
from strands.multiagent import GraphBuilder
from strands.multiagent.graph import GraphState
from strands_tools.a2a_client import A2AClientToolProvider

from sigv4_auth import SigV4HTTPXAuth

logger = logging.getLogger(__name__)


# =============================================================================
# Intent Parsing
# =============================================================================


def _parse_classifier_intent(state: GraphState) -> str:
    """Extract intent from Classifier node result.

    The classifier outputs JSON: {"intent": "BROWSE|ORDER|RECOMMEND", "reasoning": "..."}
    Parses the result text to extract the intent string with fallback handling.

    Args:
        state: Current graph execution state containing node results.

    Returns:
        Intent string: BROWSE, ORDER, or RECOMMEND. Defaults to BROWSE.
    """
    classifier_result = state.results.get("classifier_node")
    if not classifier_result:
        logger.warning("No classifier result found, defaulting to BROWSE")
        return "BROWSE"

    result_text = str(classifier_result.result)

    # Try JSON parsing first
    try:
        json_match = re.search(r'\{[^}]*"intent"\s*:\s*"(\w+)"[^}]*\}', result_text)
        if json_match:
            intent = json_match.group(1).upper()
            if intent in ("BROWSE", "ORDER", "RECOMMEND"):
                logger.info(f"Parsed intent from JSON: {intent}")
                return intent
    except Exception:
        pass

    # Fallback: keyword detection in result text
    text_upper = result_text.upper()
    for intent in ["RECOMMEND", "ORDER", "BROWSE"]:
        if intent in text_upper:
            logger.info(f"Detected intent from text: {intent}")
            return intent

    logger.warning("Could not parse intent, defaulting to BROWSE")
    return "BROWSE"


# =============================================================================
# Condition Functions for Graph Edges
# =============================================================================


def should_route_to_product(state: GraphState) -> bool:
    """Route to Product Agent for BROWSE or RECOMMEND intents."""
    intent = _parse_classifier_intent(state)
    return intent in ("BROWSE", "RECOMMEND")


def should_route_to_order(state: GraphState) -> bool:
    """Route to Order Agent for ORDER or RECOMMEND intents."""
    intent = _parse_classifier_intent(state)
    return intent in ("ORDER", "RECOMMEND")


def should_route_to_recommendation(state: GraphState) -> bool:
    """Route to Recommendation Agent for RECOMMEND intent only."""
    intent = _parse_classifier_intent(state)
    return intent == "RECOMMEND"


# =============================================================================
# Node Agent Factory
# =============================================================================


def _create_a2a_node_agent(
    name: str,
    description: str,
    system_prompt: str,
    agent_url: str,
    auth: SigV4HTTPXAuth,
    region: str,
) -> Agent:
    """Create a local agent that forwards requests to a remote A2A agent.

    Each graph node needs its own Agent instance. The agent uses
    A2AClientToolProvider to discover and call the remote agent via A2A protocol.

    Args:
        name: Agent name for identification in graph execution logs.
        description: Agent description for graph node metadata.
        system_prompt: Instructions for forwarding behavior.
        agent_url: Amazon Bedrock AgentCore invocation URL for the remote agent.
        auth: SigV4 authentication handler for AgentCore API calls.
        region: AWS region for Amazon Bedrock model invocation.

    Returns:
        Configured Agent instance ready for use as a graph node.
    """
    a2a_provider = A2AClientToolProvider(
        known_agent_urls=[agent_url],
        httpx_client_args={"auth": auth},
    )

    return Agent(
        name=name,
        description=description,
        system_prompt=system_prompt,
        model=BedrockModel(
            model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
            region_name=region,
        ),
        tools=a2a_provider.tools,
    )


# =============================================================================
# Graph Construction
# =============================================================================


def create_graph(
    classifier_url: str,
    product_url: str,
    order_url: str,
    recommendation_url: str,
):
    """Create the graph orchestrator with 4 remote agent nodes.

    Constructs a GraphBuilder DAG where each node is a local Agent that
    forwards requests to a remote A2A agent on its own AgentCore runtime.
    Conditional edges determine which agents execute based on the Classifier's
    intent output.

    Args:
        classifier_url: AgentCore invocation URL for Classifier Agent.
        product_url: AgentCore invocation URL for Product Agent.
        order_url: AgentCore invocation URL for Order Agent.
        recommendation_url: AgentCore invocation URL for Recommendation Agent.

    Returns:
        Built Graph instance ready for execution via stream_async().
    """
    session = boto3.Session()
    region = session.region_name or "us-west-2"
    credentials = session.get_credentials()

    auth = SigV4HTTPXAuth(
        credentials=credentials,
        service="bedrock-agentcore",
        region=region,
    )

    # Create node agents - each wraps a remote A2A agent
    classifier_agent = _create_a2a_node_agent(
        name="Classifier_Node",
        description="Classifies customer intent into BROWSE, ORDER, or RECOMMEND",
        system_prompt=(
            "Forward the user's message to the classifier agent using the A2A tool. "
            "Return the classifier's response exactly as received. Do not modify or "
            "interpret the response."
        ),
        agent_url=classifier_url,
        auth=auth,
        region=region,
    )

    product_agent = _create_a2a_node_agent(
        name="Product_Node",
        description="Searches product catalog via DummyJSON API",
        system_prompt=(
            "Forward the user's request to the product agent using the A2A tool. "
            "Pass along any context about what products to search for. "
            "Return the product agent's response exactly as received."
        ),
        agent_url=product_url,
        auth=auth,
        region=region,
    )

    order_agent = _create_a2a_node_agent(
        name="Order_Node",
        description="Looks up customer orders from DynamoDB",
        system_prompt=(
            "Forward the user's request to the order agent using the A2A tool. "
            "Include any customer context if available. "
            "Return the order agent's response exactly as received."
        ),
        agent_url=order_url,
        auth=auth,
        region=region,
    )

    recommendation_agent = _create_a2a_node_agent(
        name="Recommendation_Node",
        description="Generates personalized product recommendations",
        system_prompt=(
            "Forward all context to the recommendation agent using the A2A tool. "
            "Include product catalog data and order history from previous nodes. "
            "Return the recommendation agent's response exactly as received."
        ),
        agent_url=recommendation_url,
        auth=auth,
        region=region,
    )

    # Build the graph DAG
    builder = GraphBuilder()

    # Add nodes
    builder.add_node(classifier_agent, "classifier_node")
    builder.add_node(product_agent, "product_node")
    builder.add_node(order_agent, "order_node")
    builder.add_node(recommendation_agent, "recommendation_node")

    # Add conditional edges
    # Classifier routes to Product for BROWSE and RECOMMEND intents
    builder.add_edge("classifier_node", "product_node", condition=should_route_to_product)
    # Classifier routes to Order for ORDER and RECOMMEND intents
    builder.add_edge("classifier_node", "order_node", condition=should_route_to_order)
    # Product and Order both route to Recommendation for RECOMMEND intent
    builder.add_edge("product_node", "recommendation_node", condition=should_route_to_recommendation)
    builder.add_edge("order_node", "recommendation_node", condition=should_route_to_recommendation)

    # Set entry point
    builder.set_entry_point("classifier_node")

    # Set execution timeout (graph involves multiple remote A2A calls)
    builder.set_execution_timeout(300.0)  # 5 minutes max

    # Build and return the graph
    graph = builder.build()

    logger.info("Graph orchestrator created with 4 nodes and 4 edges")
    logger.info("  classifier_node -> product_node (BROWSE, RECOMMEND)")
    logger.info("  classifier_node -> order_node (ORDER, RECOMMEND)")
    logger.info("  product_node -> recommendation_node (RECOMMEND)")
    logger.info("  order_node -> recommendation_node (RECOMMEND)")

    return graph

### Application Entrypoint

The following cell creates `graph_orchestrator/app.py`:

| Section | What It Does |
|---------|---------------|
| Configuration | SSM parameter paths for 4 agent URLs |
| URL Resolution | Retrieves agent URLs from environment or SSM Parameter Store |
| Graph Initialization | Creates graph with all 4 agent URLs at module load time |
| HTTP Entrypoint | `@app.entrypoint` handler that streams graph execution events |

In [ ]:
%%writefile graph_orchestrator/app.py
"""Graph Orchestrator application deployed to Amazon Bedrock AgentCore with HTTP protocol.

Uses GraphBuilder DAG for deterministic routing of customer queries through
classifier, product, order, and recommendation agents deployed to AgentCore.

Entry point for the graph-based multi-agent e-commerce assistant. Receives HTTP
requests, executes the graph DAG, and streams results back to the caller.
"""

import json
import logging
import os

import boto3
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands.telemetry import StrandsTelemetry

from graph_agent import create_graph

# --- Logging ---

logging.basicConfig(level=logging.INFO)
logging.getLogger("strands").setLevel(logging.INFO)
logger = logging.getLogger(__name__)

StrandsTelemetry().setup_otlp_exporter()

# --- Configuration ---

SSM_CLASSIFIER_AGENT_URL = "/ecommerce-graph/classifier-agent-url"
SSM_PRODUCT_AGENT_URL = "/ecommerce-graph/product-agent-url"
SSM_ORDER_AGENT_URL = "/ecommerce-graph/order-agent-url"
SSM_RECOMMENDATION_AGENT_URL = "/ecommerce-graph/recommendation-agent-url"

app = BedrockAgentCoreApp()


# --- Helper ---


def get_agent_url(ssm_param: str, env_var: str) -> str:
    """Get agent URL from environment variable or SSM Parameter Store.

    Args:
        ssm_param: SSM parameter name for the agent URL.
        env_var: Environment variable name to check first.

    Returns:
        Agent runtime invocation URL.
    """
    if url := os.environ.get(env_var):
        return url
    ssm = boto3.client("ssm")
    response = ssm.get_parameter(Name=ssm_param, WithDecryption=True)
    return response["Parameter"]["Value"]


# --- Initialize Graph ---

logger.info("Initializing graph orchestrator...")

classifier_url = get_agent_url(SSM_CLASSIFIER_AGENT_URL, "CLASSIFIER_AGENT_URL")
product_url = get_agent_url(SSM_PRODUCT_AGENT_URL, "PRODUCT_AGENT_URL")
order_url = get_agent_url(SSM_ORDER_AGENT_URL, "ORDER_AGENT_URL")
recommendation_url = get_agent_url(SSM_RECOMMENDATION_AGENT_URL, "RECOMMENDATION_AGENT_URL")

logger.info(f"Classifier URL: {classifier_url}")
logger.info(f"Product URL: {product_url}")
logger.info(f"Order URL: {order_url}")
logger.info(f"Recommendation URL: {recommendation_url}")

graph = create_graph(classifier_url, product_url, order_url, recommendation_url)

logger.info("Graph orchestrator initialized")


# --- Entrypoint ---


@app.entrypoint
async def invoke(payload):
    """Handle incoming requests by executing the graph DAG.

    The graph automatically:
    1. Runs classifier to determine intent (BROWSE, ORDER, RECOMMEND)
    2. Routes to appropriate agents based on conditional edges
    3. Executes independent agents in parallel (Product + Order for RECOMMEND)
    4. Chains results through recommendation node when needed

    Args:
        payload: Request dictionary with 'prompt' and optional 'customer_id'.

    Yields:
        Structured event dictionaries for streaming display.
    """
    user_input = payload.get("prompt", "")
    customer_id = payload.get("customer_id", "CUST-101")

    logger.info(f"=== INCOMING REQUEST === Customer: {customer_id}")
    logger.info(f"Prompt: {user_input}")

    # Execute graph with streaming
    async for event in graph.stream_async(user_input):
        if isinstance(event, dict):
            # Node lifecycle events
            if "multi_agent_node_start" in event:
                node_id = event["multi_agent_node_start"].get("node_id", "")
                logger.info(f">> Node started: {node_id}")
                yield json.dumps({"type": "node_start", "node_id": node_id}) + "\n"

            elif "multi_agent_node_stop" in event:
                node_id = event["multi_agent_node_stop"].get("node_id", "")
                logger.info(f"<< Node completed: {node_id}")
                yield json.dumps({"type": "node_stop", "node_id": node_id}) + "\n"

            elif "multi_agent_node_stream" in event:
                # Forward inner agent streaming events
                inner = event.get("multi_agent_node_stream", {})
                node_id = inner.get("node_id", "")
                inner_event = inner.get("event", {})

                # Extract text from nested Amazon Bedrock Converse API events
                if isinstance(inner_event, dict) and "event" in inner_event:
                    bedrock_event = inner_event["event"]
                    if "contentBlockDelta" in bedrock_event:
                        delta = bedrock_event["contentBlockDelta"].get("delta", {})
                        if "text" in delta:
                            yield json.dumps({
                                "type": "text",
                                "content": delta["text"],
                                "node_id": node_id,
                            }) + "\n"

            elif "result" in event:
                # Final graph result with execution metrics
                result = event["result"]
                yield json.dumps({
                    "type": "graph_complete",
                    "total_nodes": getattr(result, "total_nodes", 0),
                    "completed_nodes": getattr(result, "completed_nodes", 0),
                    "execution_time": getattr(result, "execution_time", 0),
                }) + "\n"

    logger.info("=== GRAPH EXECUTION COMPLETE ===")


if __name__ == "__main__":
    app.run()

### SigV4 Authentication

The following cell creates `graph_orchestrator/sigv4_auth.py` for signing A2A requests to AgentCore:

In [ ]:
%%writefile graph_orchestrator/sigv4_auth.py
"""AWS Signature Version 4 authentication for httpx clients.

Provides SigV4 request signing for httpx HTTP clients to authenticate requests
to Amazon Bedrock AgentCore API endpoints. Implements proper connection header
removal and credential refresh handling required for AgentCore inter-agent calls.

Key features:
- Connection header removal to prevent SignatureDoesNotMatch errors
- Single auth_flow method with httpx sync/async dispatching
- Fresh signer creation per request for IAM role credential refresh
- Integration with A2AClientToolProvider via httpx_client_args parameter

Based on AWS sample:
awslabs/amazon-bedrock-agentcore-samples/.../streamable_http_sigv4.py
"""

from typing import Generator

import httpx
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
from botocore.credentials import Credentials


class SigV4HTTPXAuth(httpx.Auth):
    """Sign httpx requests with AWS Signature Version 4 for Amazon Bedrock AgentCore.

    Enables A2AClientToolProvider to authenticate with Amazon Bedrock AgentCore-hosted
    agents by signing HTTP requests according to AWS SigV4 protocol. Handles connection
    header removal and credential refresh required for AgentCore API calls.
    """

    def __init__(self, credentials: Credentials, service: str, region: str):
        """Initialize AWS SigV4 authentication.

        Args:
            credentials: AWS credentials from boto3.Session().get_credentials().
            service: AWS service name for signing scope (use "bedrock-agentcore").
            region: AWS region for signing scope (e.g., "us-west-2").
        """
        self.credentials = credentials
        self.service = service
        self.region = region

    def auth_flow(
        self, request: httpx.Request
    ) -> Generator[httpx.Request, httpx.Response, None]:
        """Sign HTTP request with AWS SigV4 before transmission.

        Connection header must be removed before signing. Amazon Bedrock AgentCore
        server excludes it from signature calculation, causing SignatureDoesNotMatch
        errors if included in client signature.

        Args:
            request: Outbound httpx request to sign.

        Yields:
            Signed request with AWS SigV4 authorization headers added.
        """
        headers = dict(request.headers)

        # Remove connection header to prevent SignatureDoesNotMatch error
        headers.pop("connection", None)

        aws_request = AWSRequest(
            method=request.method,
            url=str(request.url),
            data=request.content,
            headers=headers,
        )

        # Get frozen credentials to ensure latest values from IAM role auto-refresh
        frozen_credentials = self.credentials.get_frozen_credentials()

        # Create fresh signer with current credentials
        signer = SigV4Auth(frozen_credentials, self.service, self.region)

        # Sign request with AWS SigV4 protocol
        signer.add_auth(aws_request)

        # Add signature headers to original request
        request.headers.update(dict(aws_request.headers))

        yield request

In [ ]:
%%writefile graph_orchestrator/requirements.txt
strands-agents[a2a,otel]
strands-agents-tools
bedrock-agentcore
fastapi
uvicorn
boto3

---
## Step 2: Deploy to Amazon Bedrock AgentCore

The Graph Orchestrator uses the **HTTP protocol** (port 8080) instead of A2A (port 9000) because it receives requests directly from users, not from other agents.

| Parameter | Value | Purpose |
|-----------|-------|----------|
| `entrypoint` | `app.py` | Python file with BedrockAgentCoreApp entrypoint |
| `protocol` | `HTTP` | User-facing HTTP endpoint on port 8080 |
| `agent_name` | `ecommerce_graph_orchestrator` | Unique identifier for this runtime |

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

# Graph Orchestrator needs permission to invoke other AgentCore runtimes
agentcore_invoke_permissions = [
    {
        "Effect": "Allow",
        "Action": ["bedrock-agentcore:InvokeAgent"],
        "Resource": [f"arn:aws:bedrock-agentcore:{region}:{account_id}:runtime/*"],
    }
]

orchestrator_role_arn = create_agentcore_role(
    ORCHESTRATOR_ROLE_NAME, account_id, region, extra_permissions=agentcore_invoke_permissions
)
print(f"IAM Role ARN: {orchestrator_role_arn}")

In [ ]:
orchestrator_dir = NOTEBOOK_DIR / "graph_orchestrator"
os.chdir(orchestrator_dir)

orchestrator_runtime = Runtime()
orchestrator_runtime.configure(
    entrypoint="app.py",
    execution_role=orchestrator_role_arn,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=ORCHESTRATOR_AGENT_NAME,
    protocol="HTTP",
)

os.chdir(NOTEBOOK_DIR)
print(f"Runtime configured: {ORCHESTRATOR_AGENT_NAME}")

### Fix Dockerfile Permissions

AgentCore containers run as the `bedrock_agentcore` user, not root.

In [ ]:
# # Uncomment if the auto-generated Dockerfile needs permission fixes
# dockerfile_path = NOTEBOOK_DIR / "graph_orchestrator" / "Dockerfile"
# if dockerfile_path.exists():
#     content = dockerfile_path.read_text()
#     if "chmod" not in content:
#         content = content.replace(
#             "COPY . /app",
#             "COPY . /app\nRUN chmod -R 755 /app",
#         )
#         dockerfile_path.write_text(content)
#         print("Dockerfile updated with permission fix")
#     else:
#         print("Dockerfile already has permission fix")
# else:
#     print("No Dockerfile found yet -- will be created during launch")

### Launch Graph Orchestrator

`Runtime.launch()` builds the Docker image, pushes it to ECR, and creates the AgentCore runtime.

**Note:** First deployment takes 5-10 minutes.

In [ ]:
print("Launching Graph Orchestrator (this may take several minutes)...")
os.chdir(orchestrator_dir)
orchestrator_launch = orchestrator_runtime.launch(auto_update_on_conflict=True)
print(f"Orchestrator ARN: {orchestrator_launch.agent_arn}")
ORCHESTRATOR_ARN = orchestrator_launch.agent_arn
os.chdir(NOTEBOOK_DIR)

### Get Runtime URL

The Graph Orchestrator uses HTTP protocol, so its URL is the direct invocation endpoint.

In [ ]:
os.chdir(orchestrator_dir)
status_response = orchestrator_runtime.status()
status = status_response.endpoint.get("status", "")
orchestrator_url = None

print(f"Graph Orchestrator Status: {status}")

if status.upper() in ["ACTIVE", "READY"]:
    agent_runtime_arn = status_response.endpoint.get("agentRuntimeArn")
    escaped_arn = quote(agent_runtime_arn, safe="")
    orchestrator_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_arn}/invocations"
    print(f"Runtime URL: {orchestrator_url}")
else:
    print(f"Agent not ready. Current status: {status}")

os.chdir(NOTEBOOK_DIR)

In [ ]:
print("=" * 60)
print("Graph Orchestrator Deployment Summary")
print("=" * 60)
print(f"Agent Name: {ORCHESTRATOR_AGENT_NAME}")
print(f"Agent ARN: {ORCHESTRATOR_ARN}")
print(f"IAM Role: {orchestrator_role_arn}")
print(f"Runtime URL: {orchestrator_url or 'Not available - agent not ready'}")
print(f"Protocol: HTTP (port 8080)")
print("=" * 60)

---
## Step 3: Test Graph Orchestrator

Test all 3 execution paths to verify the graph DAG routes correctly:

| Test | Expected Path | Agents Invoked |
|------|---------------|----------------|
| BROWSE | Classifier -> Product | 2 |
| ORDER | Classifier -> Order | 2 |
| RECOMMEND | Classifier -> Product + Order -> Recommendation | 4 |

### Test 1: BROWSE Path

A product search query should route through Classifier -> Product only.

In [ ]:
browse_query = "Show me laptops under $1000"

payload = {
    "prompt": browse_query,
    "customer_id": "CUST-101",
}

os.chdir(orchestrator_dir)
print(f"Testing BROWSE path: '{browse_query}'")
print("Expected path: classifier_node -> product_node")
print("=" * 60)
response = orchestrator_runtime.invoke(payload, session_id=str(uuid4()))

# Parse and display streaming events
if isinstance(response, str):
    for line in response.strip().split("\n"):
        if line.strip():
            try:
                event = json.loads(line)
                event_type = event.get("type", "")
                if event_type == "node_start":
                    print(f"\n>> Executing: {event['node_id']}")
                elif event_type == "node_stop":
                    print(f"<< Completed: {event['node_id']}")
                elif event_type == "text":
                    print(event.get("content", ""), end="")
                elif event_type == "graph_complete":
                    print(f"\n{'=' * 60}")
                    print(f"Nodes: {event.get('completed_nodes', 0)}/{event.get('total_nodes', 0)}")
            except json.JSONDecodeError:
                print(line)
else:
    print(response)

os.chdir(NOTEBOOK_DIR)

### Test 2: ORDER Path

An order inquiry should route through Classifier -> Order only.

In [ ]:
order_query = "What are my recent orders?"

payload = {
    "prompt": order_query,
    "customer_id": "CUST-101",
}

os.chdir(orchestrator_dir)
print(f"Testing ORDER path: '{order_query}'")
print("Expected path: classifier_node -> order_node")
print("=" * 60)
response = orchestrator_runtime.invoke(payload, session_id=str(uuid4()))

if isinstance(response, str):
    for line in response.strip().split("\n"):
        if line.strip():
            try:
                event = json.loads(line)
                event_type = event.get("type", "")
                if event_type == "node_start":
                    print(f"\n>> Executing: {event['node_id']}")
                elif event_type == "node_stop":
                    print(f"<< Completed: {event['node_id']}")
                elif event_type == "text":
                    print(event.get("content", ""), end="")
                elif event_type == "graph_complete":
                    print(f"\n{'=' * 60}")
                    print(f"Nodes: {event.get('completed_nodes', 0)}/{event.get('total_nodes', 0)}")
            except json.JSONDecodeError:
                print(line)
else:
    print(response)

os.chdir(NOTEBOOK_DIR)

### Test 3: RECOMMEND Path (All 4 Agents)

A recommendation request should route through all agents:
1. Classifier determines RECOMMEND intent
2. Product and Order agents run **in parallel**
3. Recommendation agent receives combined context from both

In [ ]:
recommend_query = "Based on my purchase history, what products would you recommend?"

payload = {
    "prompt": recommend_query,
    "customer_id": "CUST-101",
}

os.chdir(orchestrator_dir)
print(f"Testing RECOMMEND path: '{recommend_query}'")
print("Expected path: classifier_node -> product_node + order_node (parallel) -> recommendation_node")
print("=" * 60)
response = orchestrator_runtime.invoke(payload, session_id=str(uuid4()))

if isinstance(response, str):
    for line in response.strip().split("\n"):
        if line.strip():
            try:
                event = json.loads(line)
                event_type = event.get("type", "")
                if event_type == "node_start":
                    print(f"\n>> Executing: {event['node_id']}")
                elif event_type == "node_stop":
                    print(f"<< Completed: {event['node_id']}")
                elif event_type == "text":
                    print(event.get("content", ""), end="")
                elif event_type == "graph_complete":
                    print(f"\n{'=' * 60}")
                    print(f"Nodes: {event.get('completed_nodes', 0)}/{event.get('total_nodes', 0)}")
            except json.JSONDecodeError:
                print(line)
else:
    print(response)

os.chdir(NOTEBOOK_DIR)

---
## Cleanup

Run this section to delete the Graph Orchestrator resources. To clean up all resources from the entire tutorial, run the cleanup sections in all 5 notebooks.

In [ ]:
print("Destroying Graph Orchestrator...")
os.chdir(orchestrator_dir)
try:
    orchestrator_runtime.destroy(delete_ecr_repo=True)
    print("Graph Orchestrator destroyed")
except Exception as e:
    print(f"Error: {e}")
os.chdir(NOTEBOOK_DIR)

In [ ]:
print("Cleaning up auto-generated files...")
for cleanup_file in ["Dockerfile", ".dockerignore"]:
    cleanup_path = orchestrator_dir / cleanup_file
    if cleanup_path.exists():
        cleanup_path.unlink()
        print(f"  Deleted: {cleanup_file}")

### Full Tutorial Cleanup

To clean up all resources from the entire tutorial, run these commands:

In [ ]:
# # Uncomment to delete ALL tutorial resources
#
# from bedrock_agentcore_starter_toolkit import destroy_bedrock_agentcore
# from utils import (
#     CLASSIFIER_AGENT_NAME,
#     PRODUCT_AGENT_NAME,
#     ORDER_AGENT_NAME,
#     RECOMMENDATION_AGENT_NAME,
#     ORCHESTRATOR_AGENT_NAME,
#     SSM_CLASSIFIER_AGENT_URL,
#     SSM_PRODUCT_AGENT_URL,
#     SSM_ORDER_AGENT_URL,
#     SSM_RECOMMENDATION_AGENT_URL,
#     SSM_ORDERS_TABLE,
#     DYNAMODB_TABLE_NAME,
#     delete_orders_table,
# )
#
# # Destroy all AgentCore runtimes
# for agent_name in [
#     ORCHESTRATOR_AGENT_NAME,
#     CLASSIFIER_AGENT_NAME,
#     PRODUCT_AGENT_NAME,
#     ORDER_AGENT_NAME,
#     RECOMMENDATION_AGENT_NAME,
# ]:
#     try:
#         destroy_bedrock_agentcore(agent_name=agent_name, delete_ecr_repo=True)
#         print(f"Destroyed: {agent_name}")
#     except Exception as e:
#         print(f"Error destroying {agent_name}: {e}")
#
# # Delete all SSM parameters
# ssm = boto3.client("ssm", region_name=region)
# for param in [
#     SSM_CLASSIFIER_AGENT_URL,
#     SSM_PRODUCT_AGENT_URL,
#     SSM_ORDER_AGENT_URL,
#     SSM_RECOMMENDATION_AGENT_URL,
#     SSM_ORDERS_TABLE,
# ]:
#     try:
#         ssm.delete_parameter(Name=param)
#         print(f"Deleted SSM: {param}")
#     except Exception as e:
#         print(f"Error: {e}")
#
# # Delete DynamoDB table
# try:
#     result = delete_orders_table(table_name=DYNAMODB_TABLE_NAME, region=region)
#     print(f"DynamoDB: {result['message']}")
# except Exception as e:
#     print(f"Error: {e}")